# Step 0 · Texts of the candidate families

For every family of the candidate pool (`in_sector` or `to_review`, from `02_candidate_pool`), we download **one English abstract** and the title of the same application, once. The next steps (keyword expansion with TF-IDF, semantic filter with Patent SBERT) read this file instead of querying PATSTAT again.

Which abstract: among the applications of the family with an English abstract, the first in the order **EP > WO > US > any other office** (EP and WO abstracts are usually the cleanest English text). The choice is made inside the database; family ids are sent in chunks of 5,000, as in Tutorial 1, notebook 11.

Families with no English abstract cannot be scored by the language model: we count them and report them.

**Before running:** run `02_candidate_pool` with the same `ENV`, so that `outputs/step0/candidates_{ENV}.parquet` is up to date.

In [ ]:
%pip install -q pyyaml

In [ ]:
from pathlib import Path

import pandas as pd

from epo.tipdata.patstat import PatstatClient

from src.sector import family_english_texts, load_config

ENV = "TEST"                       # same environment as the candidate pool
OUTPUT_DIR = Path("outputs/step0")
CHUNK = 5000
pd.set_option("display.max_colwidth", 120)

cfg = load_config()
patstat = PatstatClient(env=ENV)
db = patstat.orm()


def read_table(stem):
    """Read outputs/step0/<stem>_<ENV> as parquet, or as CSV if parquet was not available."""
    path = OUTPUT_DIR / f"{stem}_{ENV}.parquet"
    return pd.read_parquet(path) if path.exists() else pd.read_csv(path.with_suffix(".csv"))


cand = read_table("candidates")
if "to_review" not in cand.columns:
    raise RuntimeError(f"candidates_{ENV} was made by an old version of 02_candidate_pool: "
                       f"pull the latest code and run 02_candidate_pool with ENV = \"{ENV}\" first.")
pool = cand[cand["in_sector"] | cand["to_review"]][["docdb_family_id", "year", "in_sector", "to_review"]]
print(f"env {ENV} | {len(pool):,} families: {int(pool['in_sector'].sum()):,} in sector, {int(pool['to_review'].sum()):,} to review")

## 1. Download the abstracts

In [ ]:
ids = pool["docdb_family_id"].astype(int).tolist()
parts = []
for start in range(0, len(ids), CHUNK):
    parts.append(patstat.df(family_english_texts(db, ids[start:start + CHUNK])))
    print(f"  {min(start + CHUNK, len(ids)):,} / {len(ids):,} families queried")

texts = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
texts = pool.merge(texts, on="docdb_family_id", how="left")
texts["has_abstract"] = texts["appln_abstract"].notna()
print(f"{int(texts['has_abstract'].sum()):,} families with an English abstract")

## 2. Coverage

Share of families with an English abstract, by tier and by period. Where coverage is low, the semantic filter sees less of the sector.

In [ ]:
texts["tier"] = texts["in_sector"].map({True: "in sector", False: "to review"})
texts["period"] = pd.cut(texts["year"], bins=[1999, 2009, 2017, 2025], labels=["2000-2009", "2010-2017", "2018-2025"])

coverage = texts.groupby(["tier", "period"], observed=True)["has_abstract"].agg(families="size", with_english_abstract="sum")
coverage["share"] = (coverage["with_english_abstract"] / coverage["families"]).round(3)
print(coverage.to_string())
print()
print("source office of the abstract:")
print(texts["appln_auth"].value_counts().head(8).to_string())

In [ ]:
with_abstract = texts[texts["has_abstract"]]
with_abstract.sample(min(3, len(with_abstract)), random_state=42)[["docdb_family_id", "appln_auth", "appln_title", "appln_abstract"]]

## 3. Save

Stays on TIP (`outputs/` is git-ignored).

In [ ]:
cols = ["docdb_family_id", "year", "in_sector", "to_review", "has_abstract",
        "appln_id", "appln_auth", "appln_title", "appln_abstract"]
path = OUTPUT_DIR / f"texts_{ENV}.parquet"
try:
    texts[cols].to_parquet(path, index=False)
except ImportError:                       # no pyarrow: fall back to CSV
    path = path.with_suffix(".csv")
    texts[cols].to_csv(path, index=False)
print(f"saved {len(texts):,} families to {path}")